In [1]:
#importing
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import numpy as np
from math import exp
import torchvision
from torchvision import datasets
import torch
import pandas as pd
import os
import optuna
from PIL import Image
Image.MAX_IMAGE_PIXELS = None  # Disables the limit check
import torch.nn.functional as F
print(f"Is GPU available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Is GPU available? True
GPU Name: Tesla T4


In [2]:
batch_size = 32
learning_rate = 0.00001
num_epochs = 50
random_seed = 67

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from torch.utils.data import random_split
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Path to the new dataset
# Check your sidebar, it's likely: 
DATA_PATH = '/kaggle/input/datasets/sivarazadi/wikiart-art-movementsstyles'

# 2. Basic Transforms for your 224x224 encoders
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    
])

def pil_loader_rgb(path):
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            if img.mode == 'P':
                img = img.convert('RGBA')   # P → RGBA first
            return img.convert('RGB')       # then → RGB
    except Exception as e:
        print(f"Bad image: {path} — {e}")
        return Image.new('RGB', (224, 224)) # blank fallback

# 3. Create Dataset
full_dataset = datasets.ImageFolder(
    root=DATA_PATH, 
    transform=transform, 
    loader=pil_loader_rgb
)

total_size = len(full_dataset)
train_size = int(0.8 * total_size)
test_size = total_size - train_size   

train_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(random_seed) # For reproducibility
)
# 4. Create the Loaders (Using 4 workers for Kaggle CPUs)
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=4
)


In [3]:
def extract_Style_mean(layer):
    return torch.mean(layer , dim =(2,3))

def extract_Style_std(layer):
    return torch.std(layer , dim = (2,3))

def ADAIN(content_feat , style_mean , style_std):
    c_mean = torch.mean(content_feat , dim=(2,3),keepdim= True)
    c_std = torch.std(content_feat, dim = (2,3) , keepdim = True) + 1e-5
    s_mean = style_mean.view(style_mean.size(0), style_mean.size(1),1,1)
    s_std = (style_std+1e-5).view(style_std.size(0),style_std.size(1),1,1)
    normalized = (content_feat - c_mean) / c_std
    return normalized * s_std + s_mean


def req_loss(og_image,recon_image):
    return F.l1_loss(og_image,recon_image)
    
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(tensor.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(tensor.device)
    return tensor * std + mean

def normalize_imagenet(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(tensor.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(tensor.device)
    return (tensor - mean) / std


In [4]:
def gaussian(size, sigma): # creates a bell‑shaped curve
    gauss = torch.Tensor([exp(-(x - size//2)**2/float(2*sigma**2)) for x in range(size)])
    return gauss/gauss.sum()

def create_window(size, channel): # gaussian distribution in 2d window 
    _1D_window = gaussian(size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, size, size).contiguous())
    return window
class SSIM(nn.Module):
    def __init__(self, size=11, channel=3):
        super(SSIM, self).__init__()
        self.size = size
        self.channel = channel
        
        window_tensor = create_window(size, channel)
        
        self.register_buffer('window', window_tensor)
        
    def forward(self, img1, img2):

        C1 = 0.0001
        C2 = 0.0009
        
        mu1 = F.conv2d(img1, self.window, padding=self.size//2, groups=self.channel)
        mu2 = F.conv2d(img2, self.window, padding=self.size//2, groups=self.channel)

        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2

        sigma1_sq = F.conv2d(img1 * img1, self.window, padding=self.size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, self.window, padding=self.size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, self.window, padding=self.size//2, groups=self.channel) - mu1_mu2

        num = (2 * mu1_mu2 + C1) * (2 * sigma12 + C2)
        den = (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
        
        ssim_map = num / den
        return ssim_map.mean()

def single_view_supcon_loss(features, labels, temperature=0.07):
    features = F.normalize(features, dim=1)  

    features = features.to(device)
    labels = labels.to(device)
    batch_size = features.shape[0]

 
    logits = torch.div(
        torch.matmul(features, features.T), 
        temperature)


    logits_max, _ = torch.max(logits, dim=1, keepdim=True)
    logits = logits - logits_max.detach()

    labels = labels.view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)


    logits_mask = torch.scatter(
        torch.ones_like(mask),
        1,
        torch.arange(batch_size).view(-1, 1).to(device),
        0)

    mask = mask * logits_mask


    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

    mask_pos_pairs = mask.sum(1)
    mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, torch.ones_like(mask_pos_pairs), mask_pos_pairs)
    
    mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

    loss = -mean_log_prob_pos.mean()

    return loss

class PerceptualLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True)
        self.features = torch.nn.Sequential(*list(vgg.features)[:16]).eval()
        for p in self.parameters(): p.requires_grad = False

    def forward(self, x, y):
        return F.l1_loss(self.features(x), self.features(y))

In [5]:
import torchvision.models as models
class style_encoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        features = list(vgg.features.children())

        self.slice1 = torch.nn.Sequential(*features[:4])
        self.slice2 = torch.nn.Sequential(*features[4:9])
        self.slice3 = torch.nn.Sequential(*features[9:16])

        for params in self.parameters():
            params.requires_grad = False

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(896,500),
            torch.nn.ReLU(),
            torch.nn.Linear(500,128),
        )

    def forward(self,x):
        s1 = self.slice1(x)
        s2 = self.slice2(s1)
        s3 = self.slice3(s2)

        mu = torch.cat([extract_Style_mean(s1),
                       extract_Style_mean(s2),
                       extract_Style_mean(s3)],dim = 1)
        std = torch.cat([extract_Style_std(s1),
                       extract_Style_std(s2),
                       extract_Style_std(s3)],dim = 1)
        style = self.mlp(torch.cat([mu,std],dim =1 ))
        style = F.normalize(style)
        return mu,std,style

In [6]:
class content_encoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=64, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=64, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),

            torch.nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=128, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=128, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),

            torch.nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=256, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=256, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=256, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),

            torch.nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=512, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=512, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=512, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),

            torch.nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=1024, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=1024, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=3, stride=1, padding=1),
            torch.nn.InstanceNorm2d(num_features=1024, eps=1e-5, momentum=0.9),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),

            
        )
    def forward(self, x):
        return self.model(x)

In [7]:
class content_decoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
       
        self.block_5_4 = torch.nn.Sequential(
            torch.nn.Upsample(scale_factor=2, mode='nearest'),
            torch.nn.Conv2d(1024, 1024, 3, 1, 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(1024, 512, 3, 1, 1),
            torch.nn.ReLU(),
            torch.nn.Upsample(scale_factor=2, mode='nearest'),
            torch.nn.Conv2d(512, 512, 3, 1, 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(512, 256, 3, 1, 1) # Output: 256 channels
        )

        # PART 3: 256 Channel Block
        self.block_3 = torch.nn.Sequential(
            torch.nn.Upsample(scale_factor=2, mode='nearest'),
            torch.nn.ReLU(),
            torch.nn.Conv2d(256, 256, 3, 1, 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(256, 128, 3, 1, 1) # Output: 128 channels
        )

        # PART 2: 128 Channel Block
        self.block_2 = torch.nn.Sequential(
            torch.nn.Upsample(scale_factor=2, mode='nearest'),
            torch.nn.ReLU(),
            torch.nn.Conv2d(128, 128, 3, 1, 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(128, 64, 3, 1, 1) # Output: 64 channels
        )

        # PART 1: 64 Channel Block + Final Output
        self.block_1 = torch.nn.Sequential(
            torch.nn.Upsample(scale_factor=2, mode='nearest'),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, 3, 1, 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 3, 3, 1, 1),
            torch.nn.Sigmoid()
        )
    def forward(self, content_embedding,style_embedding_mu,style_embedding_std):
        pred = self.block_5_4(content_embedding)
        pred = ADAIN(pred,style_embedding_mu[:,192:448],style_embedding_std[:,192:448])
        pred = self.block_3(pred)
        pred = ADAIN(pred,style_embedding_mu[:,64:192],style_embedding_std[:,64:192])
        pred = self.block_2(pred)
        pred = ADAIN(pred,style_embedding_mu[:,0:64],style_embedding_std[:,0:64])
        pred_image = self.block_1(pred)
        return pred_image

In [8]:
import optuna
import time
# finding best hyperparameters with optuna
def objective(trial):

    lr_trial = trial.suggest_float("lr", 1e-8, 1e-3, log=True)
    recon_w_trial = trial.suggest_float("recon_weight", 0.01, 0.7, log=True)
    

    t_style_enc = style_encoder().to(device)
    t_cont_enc = content_encoder().to(device)
    t_cont_dec = content_decoder().to(device)
    t_style_enc = torch.nn.DataParallel(t_style_enc).to(device)
    t_cont_enc = torch.nn.DataParallel(t_cont_enc).to(device)
    t_cont_dec = torch.nn.DataParallel(t_cont_dec).to(device)
    
    t_params = list(t_style_enc.parameters()) + list(t_cont_enc.parameters()) + list(t_cont_dec.parameters())
    t_optimizer = torch.optim.Adam(t_params, lr=lr_trial)
    t_ssim_metric = SSIM().to(device)
    

    t_style_enc.train(); t_cont_enc.train(); t_cont_dec.train()
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        cont_emb = t_cont_enc(images)
        style_mu, style_std, style_vec = t_style_enc(images)
        recon_images = t_cont_dec(cont_emb, style_mu, style_std)
        recon_images_norm = normalize_imagenet(recon_images)
        
        loss = single_view_supcon_loss(style_vec, labels) + req_loss(images, recon_images_norm) * recon_w_trial
        
        t_optimizer.zero_grad()
        loss.backward()
        t_optimizer.step()
        
        if batch_idx > 100: break 
        
    with torch.no_grad():
        img_01 = denormalize(images).clamp(0, 1)
        score = t_ssim_metric(img_01, recon_images).item()
    return score

perceptual_loss = PerceptualLoss().to(device)
cont_enc = content_encoder().to(device)
style_enc =  style_encoder().to(device)
cont_dec = content_decoder().to(device)

cont_enc = torch.nn.DataParallel(cont_enc).to(device)
style_enc = torch.nn.DataParallel(style_enc).to(device)
cont_dec = torch.nn.DataParallel(cont_dec).to(device)
Params = list(style_enc.parameters()) + list(cont_enc.parameters()) + list(cont_dec.parameters())

optimizer = torch.optim.Adam(Params,lr = learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

start_epoch = 0
checkpoint_path = "/kaggle/input/datasets/nishitgupta09/alsyssa/muse_model_epoch_46.pth"  # last saved epoch

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    style_enc.load_state_dict(checkpoint['style_enc_state_dict'])
    cont_enc.load_state_dict(checkpoint['cont_enc_state_dict'])
    cont_dec.load_state_dict(checkpoint['cont_dec_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resumed from epoch {start_epoch}")

    
# if start_epoch == 0:
#     study = optuna.create_study(direction="maximize")
#     study.optimize(objective, n_trials=5)
#     best_p = study.best_params
#     print(f"Best Params: {best_p}")
    
#     learning_rate = best_p["lr"]
#     recon_weight = best_p["recon_weight"]

# Training 
start_time = time.time()
ssim_metric = SSIM().to(device)
accumulation_steps = 32
for epoch in range(start_epoch,num_epochs):
    ssim_list = []
    style_enc.train()
    cont_enc.train()
    cont_dec.train()
    total_loss = 0

    if epoch < 5:
        print(f"Epoch : {epoch} | reconstruction")
        for p in style_enc.parameters():
            p.requires_grad = False
        for p in cont_enc.parameters():
            p.requires_grad = True
        for p in cont_dec.parameters():
            p.requires_grad = True
    
    elif 5 <=epoch < 10:
        print(f"Epoch : {epoch} | style training")
        for p in style_enc.parameters():
            p.requires_grad = True
        for p in cont_enc.parameters():
            p.requires_grad = False
        for p in cont_dec.parameters():
            p.requires_grad = False
        
    else:
        print(f"Epoch : {epoch} | fine tuning")
        for p in style_enc.parameters():
            p.requires_grad = True
        for p in cont_enc.parameters():
            p.requires_grad = True
        for p in cont_dec.parameters():
            p.requires_grad = True

    for batch_idx ,(images,labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        if epoch == 5:
            for p in style_enc.module.slice3.parameters():
                
                p.requires_grad = True # unfreeze the style vector for training 
        

        cont_emb = cont_enc(images)
        style_mu, style_std, style_vec = style_enc(images)
        recon_images = cont_dec(cont_emb, style_mu, style_std)

        recon_images_norm = normalize_imagenet(recon_images)
        
        loss_recon = req_loss(images, recon_images_norm)

        loss_con = single_view_supcon_loss(style_vec, labels)
        

        loss = loss_con + loss_recon * 0.11 + perceptual_loss(images, recon_images_norm)*0.05
        loss = loss / accumulation_steps
        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0: # for gradient accumulation i.e. effective batch size = 1024
            optimizer.step()
            optimizer.zero_grad()

        with torch.no_grad():
            img_01 = denormalize(images).clamp(0, 1) # denormalize as reconstructed image is not normalized

            ssim_val = ssim_metric(img_01, recon_images)
        ssim_list.append(ssim_val.item())

        torch.nn.utils.clip_grad_norm_(Params, max_norm=1.0)

        if(batch_idx%32 == 0):
            print(f"loss_recon = {loss_recon.item():.4f} |")
            print(f"loss_con = {loss_con.item():.4f} |")
            print(f"SSIM = {ssim_val.item():.4f} |")
            print(f"batch_idx {batch_idx} completed , Loss: {loss.item():.4f} |")
            print(f"time_elapsed = {time.time()-start_time}")
            
        total_loss += loss
    

    scheduler.step(total_loss/len(train_loader))

    # saving the model parameters
    save_path = f"muse_model_epoch_{epoch}.pth"

    torch.save({
    'epoch': epoch,
    'style_enc_state_dict': style_enc.state_dict(),
    'cont_enc_state_dict': cont_enc.state_dict(),
    'cont_dec_state_dict': cont_dec.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
     }, save_path)

    print(f"Model saved to {save_path}")

    print(f"Epoch {epoch} completed. Loss: {loss.item():.4f}")
    print(f"Mean SSIM: {np.mean(ssim_list):.4f}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 196MB/s]


Resumed from epoch 47
Epoch : 47 | fine tuning
loss_recon = 0.2842 |
loss_con = 2.3999 |
SSIM = 0.5272 |
batch_idx 0 completed , Loss: 0.0778 |
time_elapsed = 7.653385877609253
loss_recon = 0.2946 |
loss_con = 2.5262 |
SSIM = 0.5167 |
batch_idx 32 completed , Loss: 0.0818 |
time_elapsed = 41.065826416015625
loss_recon = 0.2887 |
loss_con = 2.7500 |
SSIM = 0.5668 |
batch_idx 64 completed , Loss: 0.0887 |
time_elapsed = 75.38797044754028
loss_recon = 0.3176 |
loss_con = 2.8097 |
SSIM = 0.4832 |
batch_idx 96 completed , Loss: 0.0907 |
time_elapsed = 112.18563508987427
loss_recon = 0.2661 |
loss_con = 2.2929 |
SSIM = 0.5491 |
batch_idx 128 completed , Loss: 0.0743 |
time_elapsed = 152.0068280696869
loss_recon = 0.2471 |
loss_con = 3.0570 |
SSIM = 0.6024 |
batch_idx 160 completed , Loss: 0.0980 |
time_elapsed = 189.89948964118958
loss_recon = 0.2876 |
loss_con = 2.4463 |
SSIM = 0.5694 |
batch_idx 192 completed , Loss: 0.0791 |
time_elapsed = 228.5880846977234
loss_recon = 0.2705 |
loss_con 

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:1694: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  current = float(metrics)


Model saved to muse_model_epoch_47.pth
Epoch 47 completed. Loss: 0.0560
Mean SSIM: 0.5369
Epoch : 48 | fine tuning
loss_recon = 0.2880 |
loss_con = 2.6857 |
SSIM = 0.5607 |
batch_idx 0 completed , Loss: 0.0866 |
time_elapsed = 1283.574741601944
loss_recon = 0.2873 |
loss_con = 2.7792 |
SSIM = 0.5578 |
batch_idx 32 completed , Loss: 0.0895 |
time_elapsed = 1322.476022720337
loss_recon = 0.3249 |
loss_con = 2.6478 |
SSIM = 0.5001 |
batch_idx 64 completed , Loss: 0.0857 |
time_elapsed = 1360.5935969352722
loss_recon = 0.2803 |
loss_con = 3.1978 |
SSIM = 0.5558 |
batch_idx 96 completed , Loss: 0.1026 |
time_elapsed = 1399.4047937393188
loss_recon = 0.2717 |
loss_con = 2.6750 |
SSIM = 0.5431 |
batch_idx 128 completed , Loss: 0.0862 |
time_elapsed = 1437.7959821224213
loss_recon = 0.2729 |
loss_con = 3.0433 |
SSIM = 0.5595 |
batch_idx 160 completed , Loss: 0.0978 |
time_elapsed = 1476.4865217208862
loss_recon = 0.2421 |
loss_con = 2.8635 |
SSIM = 0.5951 |
batch_idx 192 completed , Loss: 0.09

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize as sk_normalize
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import numpy as np
import matplotlib.pyplot as plt
import time

def extract_style_vectors(dataset, split_name, batch_size=128, num_workers=4):
    """Extract and cache style vectors from style_encoder."""
    cache_path = f"/kaggle/input/datasets/yoyo963/vector/style_cache_{split_name} (1).pt"

    if os.path.exists(cache_path):
        cached = torch.load(cache_path, map_location="cpu")
        return cached["features"].numpy(), cached["targets"].numpy()

    image_loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    style_enc.eval()
    all_features, all_targets = [], []

    with torch.inference_mode():
        for images, targets in image_loader:
            images = images.to(device, non_blocking=True)
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    _, _, style_vec = style_enc(images)
            else:
                _, _, style_vec = style_enc(images)
            all_features.append(style_vec.detach().cpu())
            all_targets.append(targets.detach().cpu())

    feature_tensor = torch.cat(all_features, dim=0).contiguous()
    target_tensor  = torch.cat(all_targets,  dim=0).contiguous()
    torch.save({"features": feature_tensor, "targets": target_tensor}, cache_path)

    return feature_tensor.numpy(), target_tensor.numpy()


In [10]:
X_train, y_train = extract_style_vectors(train_dataset, split_name="train")
X_test,  y_test  = extract_style_vectors(test_dataset,  split_name="test")
X_train = sk_normalize(X_train, norm="l2")
X_test  = sk_normalize(X_test,  norm="l2")

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

K = 11
knn = KNeighborsClassifier(n_neighbors=K, metric="cosine", algorithm="brute", n_jobs=-1)

t0 = time.time()
knn.fit(X_train, y_train)
print(f"KNN fitted in {time.time()-t0:.2f}s  (k={K})")


RuntimeError: Parent directory /kaggle/input/datasets/yoyo963/vector does not exist.

In [ ]:
def mean_precision_at_k(actual, predicted_top_k, k):
    """Precision@k for single-label classification."""
    hits = [1 if a in p[:k] else 0 for a, p in zip(actual, predicted_top_k)]
    return float(np.mean(hits))

# Get top-K predictions for every test sample
proba = knn.predict_proba(X_test)                  # (N, n_classes)
sorted_classes = knn.classes_                      # class-index array

# Rank classes by probability (descending) top-K indices
top_k_classes = sorted_classes[np.argsort(-proba, axis=1)]  # (N, n_classes)

y_pred = top_k_classes[:, 0]   # top-1 prediction

print("=" * 50)
print(f"Test Accuracy (top-1): {accuracy_score(y_test, y_pred)*100:.2f}%")
for k_val in [1, 3, 5, 10, 20]:
    mp = mean_precision_at_k(y_test, top_k_classes, k_val)
    print(f"  Mean Precision@{k_val}: {mp*100:.2f}%")
print("=" * 50)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=sorted_classes)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title(f"KNN (k={K}) Confusion Matrix — Style Classification")
plt.tight_layout()
plt.savefig("knn_confusion_matrix.png", dpi=150)
plt.show()
print("Saved to knn_confusion_matrix.png")
